# CNN prompt-length ablation

Compare the winning text-only CNN with two variants that receive an additional input: raw token count or min-max scaled token count.

## 1. Import libraries and locate the project

In [16]:
from collections import Counter
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

working_dir = Path.cwd()
ROOT = next(
    (
        path
        for path in (working_dir, *working_dir.parents)
        if (path / "cnn" / "cnn_study.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from inside the prompt_injection workspace.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cnn.cnn_study import load_datasets

## 2. Load training and validation data

The first ablation uses raw data to match the winning CNN. The loader preserves the original prompt in both variants so both can use the same definition of token count. Cross-validation and early stopping use partitions of the training split. The separate validation split is for development evaluation after cross-validation.

In [17]:
RAW_DIR = ROOT / "data"
NORMALIZED_DIR = RAW_DIR / "normalized"
SPLITS = ("train", "validation")
VARIANTS = ("raw", "normalized")
ACTIVE_VARIANT = "raw"

datasets = {
    split: load_datasets(RAW_DIR, NORMALIZED_DIR, split)
    for split in SPLITS
}

train_rows = datasets["train"][ACTIVE_VARIANT]
validation_rows = datasets["validation"][ACTIVE_VARIANT]

## 3. Verify row and class counts

In [18]:
summary = []
for variant in VARIANTS:
    for split in SPLITS:
        rows = datasets[split][variant]
        counts = Counter(row["label"] for row in rows)
        summary.append({
            "dataset": variant,
            "split": split,
            "rows": len(rows),
            "benign": counts[0],
            "injection": counts[1],
        })

display(pd.DataFrame(summary))

,dataset,split,rows,benign,injection
0,raw,train,18909,9457,9452
1,raw,validation,1000,497,503
2,normalized,train,18830,9457,9373
3,normalized,validation,1000,497,503


## 4. Set up the text-only baseline

Load the confirmed winner for the active dataset from `cnn_study_v2_full_01`. The saved configuration provides the architecture, preprocessing, and hyperparameters for both length variants. The saved folds provide the weight-training, early-stopping, and held-out rows. Each new result will be paired with the text-only result from the same fold and seed.

In [19]:
from cnn import cnn_experiments as exp
from cnn.cnn_study import StudySettings, config_from_dict, partition_rows

RUN_NAME = "cnn_study_v2_full_01"
BASELINE_RUN_DIR = exp.ARTIFACTS_DIR / RUN_NAME
baseline_manifest = exp.read_json(BASELINE_RUN_DIR / "manifest.json")
selection = exp.read_json(BASELINE_RUN_DIR / "refit" / "selected.json")
confirmation = exp.read_json(BASELINE_RUN_DIR / "confirmation" / "results.json")

signature = baseline_manifest["run_signature"]
manifest_payload = {
    key: value for key, value in baseline_manifest.items() if key != "run_signature"
}
if exp.fingerprint(manifest_payload) != signature or baseline_manifest["mode"] != "full":
    raise ValueError("Expected an intact manifest for the full baseline study.")
if selection["run_signature"] != signature or confirmation["run_signature"] != signature:
    raise ValueError("The saved selection and confirmation results must match the baseline manifest.")

for dataset_name, data_dir in (("raw", RAW_DIR), ("normalized", NORMALIZED_DIR)):
    if exp.sha256(data_dir / "train.json") != baseline_manifest["training_sha256"][dataset_name]:
        raise ValueError(f"The local {dataset_name} training data differs from the saved study.")

if [row["source_row"] for row in train_rows] != baseline_manifest["source_rows"][ACTIVE_VARIANT]:
    raise ValueError("The loaded training rows differ from the rows used for the saved baseline.")

baseline_id = selection["selected"][ACTIVE_VARIANT]["config_id"]
baseline_entry = next(
    (
        entry for entry in confirmation["entries"]
        if entry["dataset"] == ACTIVE_VARIANT and entry["config_id"] == baseline_id
    ),
    None,
)
if baseline_entry is None or baseline_entry["status"] != "complete":
    raise ValueError(f"Missing completed baseline results for {ACTIVE_VARIANT}/{baseline_id}.")

baseline_config = config_from_dict(baseline_entry["config"])
if baseline_config.length_feature != "none":
    raise ValueError("The selected baseline must be text-only.")

saved_settings = baseline_manifest["settings"]
training_seeds = tuple(saved_settings["training_seeds"])
training_settings = StudySettings(**{**saved_settings, "training_seeds": training_seeds})

print(f"Saved baseline: {ACTIVE_VARIANT}/{baseline_id}")
print(f"Confirmed mean macro F1: {baseline_entry['macro_f1_mean']:.2%}")
display(pd.json_normalize(baseline_entry["config"]).T.rename(columns={0: "Value"}))

Saved baseline: raw/combined
Confirmed mean macro F1: 98.40%


,Value
config_id,combined
name,Combined improvements (raw)
embedding_dim,64
num_filters,512
kernel_size,5
depth,1
dropout,0.3
learning_rate,0.001
weight_decay,0.0001
batch_size,32


In [20]:
folds = baseline_manifest["folds"]
folds_by_id = {fold["fold"]: fold for fold in folds}
if len(folds_by_id) != len(folds) or len(folds) != training_settings.n_splits:
    raise ValueError("Expected one saved assignment for each baseline fold.")

fold_partitions = {
    fold_id: partition_rows(train_rows, fold)
    for fold_id, fold in folds_by_id.items()
}

saved_runs = baseline_entry["fold_results"]
baseline_fold_runs = {(run["fold"], run["seed"]): run for run in saved_runs}
expected_pairs = {(fold_id, seed) for fold_id in folds_by_id for seed in training_seeds}
if baseline_fold_runs.keys() != expected_pairs or len(baseline_fold_runs) != len(saved_runs):
    raise ValueError("Expected one saved baseline result for each fold and seed.")

for (fold_id, seed), run in baseline_fold_runs.items():
    if run["status"] != "complete" or run["identity"]["fold"] != folds_by_id[fold_id]:
        raise ValueError(f"Saved baseline result does not match fold {fold_id}, seed {seed}.")

baseline_fold_scores = pd.DataFrame(
    [
        {"fold": fold_id, "seed": seed, "macro_f1": run["metrics"]["macro_f1"]}
        for (fold_id, seed), run in sorted(baseline_fold_runs.items())
    ]
)
fold_sizes = pd.DataFrame.from_dict(
    {
        fold_id: {role: len(rows) for role, rows in parts.items()}
        for fold_id, parts in fold_partitions.items()
    },
    orient="index",
).rename_axis("fold").rename(columns={"train": "weight_training"})

macro_f1_percent = 100 * baseline_fold_scores.pivot(
    index="fold", columns="seed", values="macro_f1"
)
display(fold_sizes.join(macro_f1_percent.add_prefix("Macro F1 (%) seed ").round(3)))

,weight_training,early_stopping,heldout,Macro F1 (%) seed 42,Macro F1 (%) seed 123,Macro F1 (%) seed 2026
fold,,,,,,
1,13614,1513,3782,98.176,98.149,97.832
2,13614,1513,3782,98.440,98.308,98.334
3,13614,1513,3782,98.466,98.731,98.651
4,13614,1513,3782,98.625,98.440,98.731
5,13615,1513,3781,98.149,98.598,98.439


## 5. Prepare the token-count features

Count regex tokens on each original prompt, including punctuation and text beyond the CNN input window. Both variants use those same counts.

For each fold, fit min-max scaling on its weight-training rows. Apply that scaler to all three partitions of the fold. Values outside the training range can fall below zero or above one.

In [21]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

from cnn.length_features import length_counts

# This mode selects token counts. LengthState applies the log transform separately.
raw_token_counts = length_counts(train_rows, "log_tokens")
token_count_by_source_row = {
    row["source_row"]: count
    for row, count in zip(train_rows, raw_token_counts, strict=True)
}

In [22]:
fold_length_features = {}
fold_length_scalers = {}
length_summary = []

for fold_id, parts in fold_partitions.items():
    counts_by_role = {
        role: np.array(
            [token_count_by_source_row[row["source_row"]] for row in rows],
            dtype=np.float64,
        ).reshape(-1, 1)
        for role, rows in parts.items()
    }
    scaler = MinMaxScaler(clip=False).fit(counts_by_role["train"])
    scaled_by_role = {
        role: scaler.transform(counts).astype(np.float32)
        for role, counts in counts_by_role.items()
    }

    fold_length_scalers[fold_id] = scaler
    fold_length_features[fold_id] = {
        "raw_tokens": {
            role: counts.astype(np.float32) for role, counts in counts_by_role.items()
        },
        "minmax_tokens": scaled_by_role,
    }

    heldout_scaled = scaled_by_role["heldout"]
    length_summary.append({
        "fold": fold_id,
        "training_min_tokens": int(scaler.data_min_[0]),
        "training_max_tokens": int(scaler.data_max_[0]),
        "heldout_scaled_min": float(heldout_scaled.min()),
        "heldout_scaled_max": float(heldout_scaled.max()),
    })

display(pd.DataFrame(length_summary).set_index("fold").round(4))

,training_min_tokens,training_max_tokens,heldout_scaled_min,heldout_scaled_max
fold,,,,
1,4,7675,0.0,0.4892
2,4,7675,0.0,0.5124
3,4,7675,0.0,0.2841
4,4,3935,0.0,1.9514
5,4,7675,0.0,0.3130


## 6. Train the two length variants

Start fresh in each fold with the winning CNN's text settings and one extra classifier input. Initialize that input's weight to zero so both variants start with the same text scores. The first run fits both variants across five folds at seed 42. Completed folds are saved under `cnn/artifacts/cnn_length_ablation_v1/` and are reused when the matching run is repeated.

In [23]:
import torch

from cnn.length_ablation import run_ablation

TRAIN_SEED = training_seeds[0]
DEVICE_REQUEST = "cuda:0"
device = exp.resolve_device(DEVICE_REQUEST)
baseline_runtime = confirmation["runtime"]
torch.set_num_threads(baseline_runtime["cpu_threads"])
if exp.runtime_info(device) != baseline_runtime:
    raise ValueError("Use the same GPU and software environment as the baseline for this paired run.")

ablation_run_dir, length_fold_runs = run_ablation(
    exp.ARTIFACTS_DIR / "cnn_length_ablation_v1",
    baseline_config,
    training_settings,
    fold_partitions,
    fold_length_features,
    fold_length_scalers,
    dataset=ACTIVE_VARIANT,
    baseline_signature=baseline_manifest["run_signature"],
    seed=TRAIN_SEED,
    micro_batch_sizes={
        fold_id: baseline_fold_runs[(fold_id, TRAIN_SEED)]["preflight"]["micro_batch_size"]
        for fold_id in fold_partitions
    },
    device=device,
)

length_fold_scores = pd.DataFrame([
    {
        "variant": run["variant"],
        "fold": run["fold"],
        "seed": run["seed"],
        "macro_f1": run["metrics"]["macro_f1"],
        "best_epoch": run["best_epoch"],
    }
    for run in length_fold_runs
])
print(f"Saved fold results: {ablation_run_dir}")
display(length_fold_scores)

Reusing raw_tokens, fold 1, seed 42
Training minmax_tokens, fold 1, seed 42 on cuda:0
  epoch 1: early-stopping loss 0.12628
  epoch 2: early-stopping loss 0.09228
  epoch 3: early-stopping loss 0.07895
  epoch 4: early-stopping loss 0.06125
  epoch 5: early-stopping loss 0.05694
  epoch 6: early-stopping loss 0.04365
  epoch 7: early-stopping loss 0.05049
  epoch 8: early-stopping loss 0.04416
  saved minmax_tokens fold 1: macro F1 97.96%
Training raw_tokens, fold 2, seed 42 on cuda:0
  epoch 1: early-stopping loss 0.12210
  epoch 2: early-stopping loss 0.08029
  epoch 3: early-stopping loss 0.08083
  epoch 4: early-stopping loss 0.04226
  epoch 5: early-stopping loss 0.04384
  epoch 6: early-stopping loss 0.03603
  epoch 7: early-stopping loss 0.04335
  epoch 8: early-stopping loss 0.04229
  saved raw_tokens fold 2: macro F1 98.36%
Training minmax_tokens, fold 2, seed 42 on cuda:0
  epoch 1: early-stopping loss 0.12865
  epoch 2: early-stopping loss 0.07676
  epoch 3: early-stopping 

,variant,fold,seed,macro_f1,best_epoch
0,raw_tokens,1,42,0.982020,9
1,minmax_tokens,1,42,0.979640,6
2,raw_tokens,2,42,0.983606,6
3,minmax_tokens,2,42,0.983606,8
4,raw_tokens,3,42,0.987837,9
5,minmax_tokens,3,42,0.985457,9
6,raw_tokens,4,42,0.985457,9
7,minmax_tokens,4,42,0.984400,10
8,raw_tokens,5,42,0.984660,10
9,minmax_tokens,5,42,0.982808,8


## 7. Compare the length variants

Pair each length result with the text-only result from the same held-out fold and training seed. Macro F1 uses a decision threshold of 0.5. Differences are in percentage points. A positive difference favors the first model named in the comparison. Wins and losses use the unrounded differences.

In [24]:
def compare_length_scores(variant_runs, seeds):
    variant_scores = pd.DataFrame([
        {
            "seed": run["seed"],
            "fold": run["fold"],
            "variant": run["variant"],
            "macro_f1": run["metrics"]["macro_f1"],
        }
        for run in variant_runs
    ]).pivot(index=["seed", "fold"], columns="variant", values="macro_f1")

    text_scores = (
        baseline_fold_scores.loc[baseline_fold_scores["seed"].isin(seeds)]
        .set_index(["seed", "fold"])["macro_f1"]
        .rename("text_only")
    )
    if set(variant_scores.columns) != {"raw_tokens", "minmax_tokens"}:
        raise ValueError("Expected both the raw and min-max token-count results.")

    paired = text_scores.to_frame().join(variant_scores, how="outer").sort_index()
    expected_pairs = {(seed, fold) for seed in seeds for fold in fold_partitions}
    if set(paired.index) != expected_pairs or paired.isna().to_numpy().any():
        raise ValueError("Expected all three model scores for each requested seed and fold.")

    scores_pct = 100 * paired[["text_only", "raw_tokens", "minmax_tokens"]].rename(columns={
        "text_only": "Text-only (%)",
        "raw_tokens": "Raw count (%)",
        "minmax_tokens": "Min-max count (%)",
    })
    deltas_pp = 100 * pd.DataFrame({
        "Raw vs text-only (pp)": paired["raw_tokens"] - paired["text_only"],
        "Min-max vs text-only (pp)": paired["minmax_tokens"] - paired["text_only"],
        "Raw vs min-max (pp)": paired["raw_tokens"] - paired["minmax_tokens"],
    })
    ties = deltas_pp.eq(0)
    summary = pd.DataFrame({
        "Mean difference (pp)": deltas_pp.mean(),
        "Wins": ((deltas_pp > 0) & ~ties).sum(),
        "Ties": ties.sum(),
        "Losses": ((deltas_pp < 0) & ~ties).sum(),
    }).rename_axis("Comparison")
    return scores_pct, deltas_pp, summary

paired_scores_pct, paired_deltas_pp, paired_summary = compare_length_scores(
    length_fold_runs, (TRAIN_SEED,),
)
display(paired_scores_pct.join(paired_deltas_pp).round(4))
display(paired_summary.round(3))

Text-only (%)  Raw count (%)  Min-max count (%)  Raw vs text-only (pp)  Min-max vs text-only (pp)  Raw vs min-max (pp)
seed fold                                                                                                                        
42   1           98.1756        98.2020            97.9640                 0.0264                    -0.2115               0.2380
     2           98.4400        98.3606            98.3606                -0.0794                    -0.0793              -0.0001
     3           98.4664        98.7837            98.5457                 0.3173                     0.0793               0.2380
     4           98.6251        98.5457            98.4400                -0.0793                    -0.1851               0.1057
     5           98.1486        98.4660            98.2808                 0.3174                     0.1322               0.1852

,Mean difference (pp),Wins,Ties,Losses
Comparison,,,,
Raw vs text-only (pp),0.100,3,0,2
Min-max vs text-only (pp),-0.053,2,0,3
Raw vs min-max (pp),0.153,4,0,1


### Confirm with additional seeds

Review the fold-by-fold differences before enabling confirmation. Setting `RUN_CONFIRMATION` to `True` trains both variants with seeds 123 and 2026 on the same folds, and reuses completed runs. The confirmation table reports mean changes within each seed. The same folds are reused across seeds, so the combined win, tie, and loss counts are descriptive.

In [25]:
RUN_CONFIRMATION = False

if RUN_CONFIRMATION:
    confirmation_length_runs = list(length_fold_runs)
    for seed in training_seeds:
        if seed == TRAIN_SEED:
            continue
        _, seed_runs = run_ablation(
            exp.ARTIFACTS_DIR / "cnn_length_ablation_v1",
            baseline_config,
            training_settings,
            fold_partitions,
            fold_length_features,
            fold_length_scalers,
            dataset=ACTIVE_VARIANT,
            baseline_signature=baseline_manifest["run_signature"],
            seed=seed,
            micro_batch_sizes={
                fold: baseline_fold_runs[(fold, seed)]["preflight"]["micro_batch_size"]
                for fold in fold_partitions
            },
            device=device,
        )
        confirmation_length_runs.extend(seed_runs)

    confirmation_scores_pct, confirmation_deltas_pp, confirmation_summary = compare_length_scores(
        confirmation_length_runs, training_seeds,
    )
    display(confirmation_scores_pct.join(confirmation_deltas_pp).round(4))
    display(confirmation_deltas_pp.groupby(level="seed").mean().round(3))
    display(confirmation_summary.round(3))

## 8. Refit both variants on the full training split

Use the median best epoch from the seed-42 folds for each variant. Rebuild the vocabulary from the full training split and fit the min-max scaler on that same split. The text-only comparator is the winning CNN already refit by the earlier study.

In [26]:
from statistics import median

from cnn.length_ablation_test import evaluate_length_test, refit_length_models

refit_epochs = {
    variant: int(median([
        run["best_epoch"] for run in length_fold_runs if run["variant"] == variant
    ]))
    for variant in ("raw_tokens", "minmax_tokens")
}
display(pd.Series(refit_epochs, name="Refit epochs").to_frame())

refit_dir, refit_checkpoints = refit_length_models(
    exp.ARTIFACTS_DIR / "cnn_length_ablation_v1",
    baseline_config,
    train_rows,
    refit_epochs,
    dataset=ACTIVE_VARIANT,
    baseline_signature=baseline_manifest["run_signature"],
    seed=TRAIN_SEED,
    micro_batch_size=selection["models"][f"{ACTIVE_VARIANT}_selected"]["micro_batch_size"],
    device=device,
)

,Refit epochs
raw_tokens,9
minmax_tokens,8


Reusing refit for raw_tokens
Reusing refit for minmax_tokens


## 9. Compare on the test benchmark

Use the fixed 0.5 decision threshold. The full test table compares the two length variants with the winning text-only CNN on the active dataset. The common table adds the other models with saved scores on rows present in both dataset variants and disjoint from training under the earlier study's overlap rule. The benchmark was evaluated previously.

In [27]:
test_report_path, length_test_report = evaluate_length_test(
    refit_dir,
    refit_checkpoints,
    baseline_dir=BASELINE_RUN_DIR,
    baseline_manifest=baseline_manifest,
    data_dirs={"raw": RAW_DIR, "normalized": NORMALIZED_DIR},
    dataset=ACTIVE_VARIANT,
    device=device,
)
test_metrics = pd.DataFrame(length_test_report["reports"])
test_metrics["predicted_injections"] = test_metrics["true_positives"] + test_metrics["false_positives"]
test_columns = {
    "model": "Model",
    "rows": "Rows",
    "macro_f1": "Macro F1 (%)",
    "injection_f1": "Injection F1 (%)",
    "injection_recall": "Injection recall (%)",
    "false_positive_rate": "Benign FPR (%)",
    "injection_count": "Actual injections",
    "true_positives": "Injections caught",
    "predicted_injections": "Predicted injections",
    "false_positives": "False positives",
    "false_negatives": "Missed injections",
}
percent_metrics = ["macro_f1", "injection_f1", "injection_recall", "false_positive_rate"]
for split, title in (
    ("test_full_known_benchmark", "Full test split"),
    ("test_common_disjoint_known_benchmark", "Common training-disjoint test rows"),
):
    comparison = test_metrics.loc[test_metrics["split"].eq(split), list(test_columns)].copy()
    comparison = comparison.sort_values("macro_f1", ascending=False)
    comparison[percent_metrics] *= 100
    print(title)
    display(comparison.rename(columns=test_columns).round(3).reset_index(drop=True))

print(f"Saved test report: {test_report_path}")

,Model,Rows,Macro F1 (%),Injection F1 (%),Injection recall (%),Benign FPR (%),Actual injections,Injections caught,Predicted injections,False positives,Missed injections
0,raw_selected,23516,75.236,64.082,63.845,13.488,6486,4141,6438,2297,2345
1,raw_tokens,23516,74.561,62.802,61.224,12.854,6486,3971,6160,2189,2515
2,minmax_tokens,23516,70.800,60.767,72.310,25.015,6486,4690,8950,4260,1796


,Model,Rows,Macro F1 (%),Injection F1 (%),Injection recall (%),Benign FPR (%),Actual injections,Injections caught,Predicted injections,False positives,Missed injections
0,raw_selected,23358,75.181,64.092,63.845,13.602,6486,4141,6436,2295,2345
1,raw_tokens,23358,74.506,62.812,61.224,12.962,6486,3971,6158,2187,2515
2,minmax_tokens,23358,70.936,61.004,72.310,24.893,6486,4690,8890,4200,1796
3,raw_reference,23358,68.988,57.508,65.217,23.678,6486,4230,8225,3995,2256
4,normalized_selected,23358,66.893,54.287,60.530,24.016,6486,3926,7978,4052,2560
5,normalized_log_characters,23358,66.618,62.755,99.923,45.567,6486,6481,14169,7688,5
6,raw_log_characters,23358,66.442,62.625,99.938,45.833,6486,6482,14215,7733,4
7,normalized_reference,23358,65.343,54.871,68.949,31.662,6486,4472,9814,5342,2014
8,normalized_log_tokens,23358,63.285,60.320,99.846,50.439,6486,6476,14986,8510,10
9,raw_log_tokens,23358,62.978,60.119,99.892,50.907,6486,6479,15068,8589,7


Full test split
Common training-disjoint test rows
Saved test report: cnn\artifacts\cnn_length_ablation_v1\raw\refit_fa3a9a80c0907781\test_dd50a63bc2b0b84e.json


In [28]:
lower, middle, upper = (int(value) for value in length_test_report["training_length_quartiles"])
band_names = {
    1: f"< {lower}",
    2: f"{lower}?{middle - 1}",
    3: f"{middle}?{upper - 1}",
    4: f"? {upper}",
}
length_test_bands = pd.DataFrame(length_test_report["length_bands"])
length_test_bands["Prompt tokens"] = length_test_bands["length_band"].map(band_names)
band_metrics = ["injection_recall", "false_positive_rate"]
length_test_bands[band_metrics] *= 100
display(
    length_test_bands[
        ["model", "Prompt tokens", "benign_rows", "injection_rows", *band_metrics]
    ].rename(columns={
        "model": "Model",
        "benign_rows": "Benign prompts",
        "injection_rows": "Injection prompts",
        "injection_recall": "Injection recall (%)",
        "false_positive_rate": "Benign FPR (%)",
    }).round(3)
)

,Model,Prompt tokens,Benign prompts,Injection prompts,Injection recall (%),Benign FPR (%)
0,raw_selected,< 23,4931,0,NaN,0.203
1,raw_selected,23?64,3715,20,45.000,14.966
2,raw_selected,65?121,3769,1456,77.404,23.322
3,raw_selected,? 122,4457,5010,59.980,19.071
4,raw_tokens,< 23,4931,0,NaN,0.243
5,raw_tokens,23?64,3715,20,50.000,14.643
6,raw_tokens,65?121,3769,1456,71.841,24.038
7,raw_tokens,? 122,4457,5010,58.184,16.267
8,minmax_tokens,< 23,4931,0,NaN,0.284
9,minmax_tokens,23?64,3715,20,70.000,19.408


### What the test results show

On the full raw test set at threshold 0.5, the text-only CNN had the highest macro F1 (75.24%). Raw token count produced 108 fewer false positives and missed 170 more injections. Min-max token count had the highest injection recall (72.31%), catching 4,690 of 6,486 injections, 549 more than the text-only CNN. It flagged 8,950 prompts, including 4,260 benign prompts. That is 1,963 more false positives than the text-only CNN. A higher alert count includes both caught injections and false alarms.